In [12]:
# Nombres y apellidos completos: Yuliana Orihuela Lazo
# Código de matrícula: 2024200514G
# Tema y número del temario: Perpetuidades y valuación de acciones con
#                             dividendo estable en la BVL -Tema 27
# Fecha de extracción: 22/09/2026
"""
01_extraccion_api.py
Vía 1 (API) del proyecto de Finanzas I. Descarga precios diarios y dividendos
de 10 emisores de la Bolsa de Valores de Lima (BVL), más dos series
auxiliares para el modelo CAPM (Ke = Rf + Beta x ERP):
  - EPU (iShares MSCI Peru ETF): proxy del mercado peruano, para calcular
    el beta de cada emisor por regresión de retornos diarios.
  - ^TNX (rendimiento del Tesoro de EE.UU. a 10 años): tasa libre de
    riesgo (Rf). Es un rendimiento en %, no un precio: no se le calculan
    retornos, se usa su cierre directamente (dividido entre 100).
Todo se descarga mediante la librería `yfinance`, que consume la API
pública de gráficos de Yahoo Finance.

ENDPOINT DECLARADO
-------------------
`yfinance` no expone una URL editable: internamente llama al endpoint
`https://query1.finance.yahoo.com/v8/finance/chart/<ticker>` (precios OHLCV)
y a `.../v10/finance/quoteSummary/<ticker>?modules=... ` (acciones
corporativas, incluidos los dividendos), con los parámetros que se declaran
más abajo (start, end, interval). Se documenta aquí porque la consigna exige
declarar el endpoint, aunque la petición HTTP la arme la propia librería.

PARÁMETROS DECLARADOS (constantes, no fechas dinámicas tipo "hoy")
-------------------------------------------------------------------
FECHA_INICIO, FECHA_CORTE, INTERVALO: ver sección 1.

MANEJO DE ERRORES
-------------------
Cada ticker se descarga dentro de un try/except independiente: un error en
un emisor no detiene la extracción de los demás. `yfinance` no expone el
código de respuesta HTTP de cada solicitud (la librería lo abstrae), así
que el log registra en su lugar: éxito o fracaso, número de filas obtenidas
y el mensaje de excepción exacto cuando falla. Esa es la evidencia
verificable que sí se puede declarar con honestidad.

GUARDADO DEL CRUDO
--------------------
El resultado de cada descarga se concatena y se guarda TAL COMO LO ENTREGA
YAHOO, sin editar valores (los dividendos de los tickers .LM llegan como
texto, p. ej. "0.377 PEN": aquí se guardan así; la conversión a número
ocurre recién en 03_limpieza_datos.py, nunca en este script ni a mano).

Instalación:  pip install yfinance pandas
Ejecución:    python 01_extraccion_api.py   (ejecutar desde /codigo)
Salidas:      ../datos_crudos/datos_crudos_<CODIGO_MATRICULA>.csv
              ../log_ejecucion.txt (se agrega una línea por corrida)
"""

import sys
import time
import traceback
from datetime import datetime
from pathlib import Path

import pandas as pd
import yfinance as yf

# ----------------------------------------------------------------------------
# 0. IDENTIFICACIÓN Y RUTAS
# ----------------------------------------------------------------------------
CODIGO_MATRICULA = "2024200514G"

# Detecta si corre como archivo .py (tiene __file__) o pegado en una celda
# de Jupyter/Colab (no tiene __file__): en ese caso usa la carpeta actual.
try:
    DIR_CODIGO = Path(__file__).resolve().parent     # .../codigo
    DIR_PROYECTO = DIR_CODIGO.parent                 # carpeta N.3 completa
except NameError:
    DIR_ACTUAL = Path.cwd()
    if DIR_ACTUAL.name == "2024200514":
        DIR_PROYECTO = DIR_ACTUAL.parent
    else:
        # Notebook corriendo fuera de /codigo: se usa la carpeta actual
        # como raíz del proyecto (crea datos_crudos/ y log aquí mismo).
        DIR_PROYECTO = DIR_ACTUAL

DIR_CRUDOS = DIR_PROYECTO / "datos_crudos"
LOG_PATH = DIR_PROYECTO / "log_ejecucion.txt"

DIR_CRUDOS.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# 1. PARÁMETROS CONGELADOS (la consigna prohíbe fechas dinámicas tipo "hoy")
# ----------------------------------------------------------------------------
FECHA_INICIO = "2018-01-01"
FECHA_CORTE = "2025-12-31"      # último día que se INCLUYE
INTERVALO = "1d"
PAUSA_SEG = 1.5                  # pausa entre solicitudes (buena práctica)

# ----------------------------------------------------------------------------
# 2. LOS 10 EMISORES VALIDADOS EN 00_validacion_tickers.py
#    (emisor, nemónico BVL, ticker exacto de yfinance)
# ----------------------------------------------------------------------------
EMISORES = [
    ("Southern Copper Corporation",                    "SCCO",     "SCCO"),
    ("Credicorp Ltd.",                                  "BAP",      "BAP"),
    ("Cementos Pacasmayo S.A.A.",                       "CPACASC1", "CPACASC1.LM"),
    ("Ferreycorp S.A.A.",                               "FERREYC1", "FERREYC1.LM"),
    ("UNACEM Corp S.A.A.",                              "UNACEMC1", "UNACEMC1.LM"),
    ("Alicorp S.A.A.",                                  "ALICORC1", "ALICORC1.LM"),
    ("Unión de Cervecerías Peruanas Backus y Johnston", "BACKUSI1", "BACKUSI1.LM"),
    ("Banco de Crédito del Perú",                       "CREDITC1", "CREDITC1.LM"),
    ("Luz del Sur S.A.A.",                              "LUSURC1",  "LUSURC1.LM"),
    ("Banco BBVA Perú",                                 "BBVAC1",   "BBVAC1.LM"),
]

# Series auxiliares para el modelo CAPM (Ke = Rf + Beta x ERP):
#   - EPU: proxy del mercado peruano, para calcular el beta de cada emisor
#     por regresión de retornos diarios (acción vs. índice).
#   - ^TNX: rendimiento del Tesoro de EE.UU. a 10 años, usado como tasa
#     libre de riesgo (Rf). OJO: ^TNX ya es un rendimiento en % (ej. 4.487
#     significa 4.487%), no un precio. No se le calculan "retornos"; su
#     valor de cierre se usa directamente (dividido entre 100) como Rf.
AUXILIARES = [
    ("Índice de mercado (proxy Perú)", "EPU_INDICE", "EPU"),
    ("Tasa libre de riesgo (Tesoro EE.UU. 10 años)", "TNX_RF", "^TNX"),
]


# ----------------------------------------------------------------------------
# 3. FUNCIONES
# ----------------------------------------------------------------------------
def fin_exclusivo(fecha_corte):
    """yfinance trata `end` como exclusivo: se suma 1 día para incluirlo."""
    return (pd.Timestamp(fecha_corte) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


def registrar_log(lineas):
    """Agrega líneas al log de ejecución (no lo sobrescribe entre corridas)."""
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        for linea in lineas:
            f.write(linea + "\n")


def descargar_ticker(nombre, nemonico, ticker, categoria="Emisor BVL"):
    """
    Descarga el histórico diario crudo de un ticker (precios + dividendos)
    desde el endpoint de Yahoo Finance vía yfinance.history().
    Devuelve (DataFrame crudo o None, línea de log).
    """
    marca = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    try:
        t = yf.Ticker(ticker)
        h = t.history(
            start=FECHA_INICIO,
            end=fin_exclusivo(FECHA_CORTE),
            interval=INTERVALO,
            auto_adjust=False,   # se conservan Close y Adj Close por separado
            actions=True,        # incluye Dividends y Stock Splits
        )

        if h is None or h.empty:
            linea = (f"{marca} | {ticker:<14} | FALLO | 0 filas | "
                      f"sin datos en el periodo declarado")
            print("  ", linea)
            return None, linea

        # Quitar el huso horario del índice: es una representación, no un
        # cambio de valores. Los datos (Open, High, Low, Close, Dividends,
        # tal cual llegan de Yahoo) no se tocan.
        if getattr(h.index, "tz", None) is not None:
            h.index = h.index.tz_localize(None)

        h = h.reset_index()                 # la fecha pasa a ser columna
        h = h.rename(columns={h.columns[0]: "Date"})
        h.insert(0, "Ticker", ticker)
        h.insert(0, "Nemonico_BVL", nemonico)
        h.insert(0, "Emisor", nombre)
        h.insert(0, "Categoria", categoria)

        linea = (f"{marca} | {ticker:<14} | EXITO | {len(h)} filas | "
                  f"{FECHA_INICIO} -> {FECHA_CORTE}")
        print("  ", linea)
        return h, linea

    except Exception as e:
        detalle = f"{type(e).__name__}: {e}"
        linea = f"{marca} | {ticker:<14} | FALLO | 0 filas | {detalle}"
        print("  ", linea)
        # Traza completa solo a la consola, para depurar sin ensuciar el log
        traceback.print_exc(file=sys.stdout)
        return None, linea


# ----------------------------------------------------------------------------
# 4. EJECUCIÓN
# ----------------------------------------------------------------------------
def main():
    print(f"yfinance {yf.__version__} | Python {sys.version.split()[0]}")
    print(f"Ventana: {FECHA_INICIO} -> {FECHA_CORTE} | Intervalo: {INTERVALO}\n")

    lineas_log = [
        "=" * 78,
        f"Corrida: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | "
        f"Código de matrícula: {CODIGO_MATRICULA}",
    ]

    crudos, fallidos = [], []
    for nombre, nemonico, ticker in EMISORES:
        print(f"Descargando {ticker} ({nombre})...")
        df, linea = descargar_ticker(nombre, nemonico, ticker, categoria="Emisor BVL")
        lineas_log.append(linea)
        if df is not None:
            crudos.append(df)
        else:
            fallidos.append(ticker)
        time.sleep(PAUSA_SEG)

    print("\n--- Series auxiliares para el modelo CAPM (índice y Rf) ---")
    for nombre, nemonico, ticker in AUXILIARES:
        print(f"Descargando {ticker} ({nombre})...")
        df, linea = descargar_ticker(nombre, nemonico, ticker, categoria="Auxiliar CAPM")
        lineas_log.append(linea)
        if df is not None:
            crudos.append(df)
        else:
            fallidos.append(ticker)
        time.sleep(PAUSA_SEG)

    if not crudos:
        print("\nNingún ticker devolvió datos. Revisa tu conexión o los "
              "tickers en EMISORES. No se generó archivo de salida.")
        registrar_log(lineas_log + ["Resultado: SIN DATOS, no se guardó CSV"])
        return

    crudo_final = pd.concat(crudos, ignore_index=True)
    nombre_archivo = f"datos_crudos_{CODIGO_MATRICULA}.csv"
    ruta_salida = DIR_CRUDOS / nombre_archivo
    crudo_final.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

    resumen = (f"Guardado: {nombre_archivo} | {len(crudo_final)} filas totales | "
               f"{len(crudos)}/{len(EMISORES) + len(AUXILIARES)} series OK "
               f"({len(EMISORES)} emisores + {len(AUXILIARES)} auxiliares)")
    print("\n" + "=" * 78)
    print(resumen)
    if fallidos:
        print("Fallaron:", ", ".join(fallidos))
        print("Si el bloqueo persiste, documenta en incidencias_fuente.md y "
              "solicita sustitución al docente con 5 días de anticipación.")

    lineas_log.append(resumen)
    if fallidos:
        lineas_log.append("Fallaron: " + ", ".join(fallidos))
    registrar_log(lineas_log)
    print(f"\nLog actualizado en: {LOG_PATH}")
    print(f"Archivo crudo en:    {ruta_salida}")


if __name__ == "__main__":
    main()

yfinance 0.2.66 | Python 3.13.15
Ventana: 2018-01-01 -> 2025-12-31 | Intervalo: 1d

Descargando SCCO (Southern Copper Corporation)...
   2026-09-24 14:39:30 | SCCO           | EXITO | 2011 filas | 2018-01-01 -> 2025-12-31
Descargando BAP (Credicorp Ltd.)...
   2026-09-24 14:39:31 | BAP            | EXITO | 2011 filas | 2018-01-01 -> 2025-12-31
Descargando CPACASC1.LM (Cementos Pacasmayo S.A.A.)...
   2026-09-24 14:39:33 | CPACASC1.LM    | EXITO | 2002 filas | 2018-01-01 -> 2025-12-31
Descargando FERREYC1.LM (Ferreycorp S.A.A.)...
   2026-09-24 14:39:34 | FERREYC1.LM    | EXITO | 2000 filas | 2018-01-01 -> 2025-12-31
Descargando UNACEMC1.LM (UNACEM Corp S.A.A.)...
   2026-09-24 14:39:36 | UNACEMC1.LM    | EXITO | 2002 filas | 2018-01-01 -> 2025-12-31
Descargando ALICORC1.LM (Alicorp S.A.A.)...
   2026-09-24 14:39:38 | ALICORC1.LM    | EXITO | 2001 filas | 2018-01-01 -> 2025-12-31
Descargando BACKUSI1.LM (Unión de Cervecerías Peruanas Backus y Johnston)...
   2026-09-24 14:39:39 | BACKUS

In [11]:
# Nombres y apellidos completos: Yuliana Orihuela Lazo
# Código de matrícula: 2024200514G
# Tema y número del temario: Perpetuidades y valuación de acciones con
#                             dividendo estable en la BVL-Tema 27
# Fecha de extracción: 24/09/2026
"""
03_limpieza_datos.py
Depuración, tipificación y preparación de la base extraída en
01_extraccion_api.py. Aquí solo se PREPARAN los datos; el cálculo de Ke,
valor teórico y brecha (el resultado del modelo) va en 04_analisis.py.

QUÉ HACE ESTE SCRIPT
-----------------------
1. Lee el crudo (datos_crudos_<CODIGO>.csv) y lo separa en dos grupos
   usando la columna Categoria: los 10 emisores de la BVL, y las 2 series
   auxiliares del modelo CAPM (EPU = índice de mercado, ^TNX = tasa libre
   de riesgo).
2. Limpia Dividends: en los tickers .LM llega como texto con la moneda
   incluida (ej. "0.377 PEN"); aquí se convierte a número. El crudo NO se
   toca; esta limpieza ocurre solo sobre una copia en memoria.
3. Calcula el retorno diario de cada emisor y del índice EPU:
       retorno_t = (Close_t / Close_(t-1)) - 1
   Nota: NO se le calculan retornos a ^TNX. ^TNX ya es un rendimiento en
   porcentaje (ej. 4.136 = 4.136%), no un precio; su Close se usa
   directamente (dividido entre 100) como tasa libre de riesgo diaria.
4. Une cada emisor con el retorno de EPU por la llave común (Date), y
   corre una regresión lineal simple (retorno_emisor ~ retorno_EPU) para
   obtener el beta de cada emisor. Esto queda en un archivo aparte,
   beta_emisores_<CODIGO>.csv, porque es un resumen (1 fila por emisor),
   no un panel diario.
5. Trata datos faltantes: un NaN el primer día de cada serie (no hay día
   anterior para calcular el retorno) se descarta, no se rellena con cero
   ni se inventa. Los días sin negociación (feriados) ya vienen alineados
   porque Yahoo solo entrega días con actividad.
6. Guarda el panel diario limpio en datos_procesados_<CODIGO>.csv, con
   Rf y el retorno de EPU ya unidos por fecha (llave común: Date).

Instalación:  pip install pandas numpy scipy
Ejecución:    python 03_limpieza_datos.py   (ejecutar desde /codigo)
Salidas:      ../datos_procesados/datos_procesados_<CODIGO>.csv
              ../datos_procesados/beta_emisores_<CODIGO>.csv
              ../log_ejecucion.txt (se agrega una línea)
"""

import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# ----------------------------------------------------------------------------
# 0. RUTAS (mismo patrón que 01 y 02: funciona como .py o en notebook)
# ----------------------------------------------------------------------------
CODIGO_MATRICULA = "2024200514G"

try:
    DIR_CODIGO = Path(__file__).resolve().parent
    DIR_PROYECTO = DIR_CODIGO.parent
except NameError:
    DIR_ACTUAL = Path.cwd()
    DIR_PROYECTO = DIR_ACTUAL.parent if DIR_ACTUAL.name == "codigo" else DIR_ACTUAL

DIR_CRUDOS = DIR_PROYECTO / "datos_crudos"
DIR_PROCESADOS = DIR_PROYECTO / "datos_procesados"
LOG_PATH = DIR_PROYECTO / "log_ejecucion.txt"
DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)

RUTA_CRUDO = DIR_CRUDOS / f"datos_crudos_{CODIGO_MATRICULA}.csv"

# Mínimo de observaciones superpuestas (emisor vs. EPU) para confiar en el
# beta. Con menos que esto, la regresión es sensible a ruido y se marca.
MIN_OBS_BETA = 250   # aprox. 1 año de ruedas


# ----------------------------------------------------------------------------
# 1. FUNCIONES
# ----------------------------------------------------------------------------
def limpiar_dividendo(x):
    """
    Convierte un dividendo crudo a número float.
    Acepta números ya limpios, vacíos, y texto como "0.377 PEN" o "0,09".
    Devuelve 0.0 si no hubo dividendo ese día, o None si el texto no se
    pudo interpretar (para que quede visible y no se pierda en silencio).
    """
    if x is None:
        return 0.0
    if isinstance(x, (int, float)):
        return 0.0 if pd.isna(x) else float(x)
    texto = str(x).strip()
    if texto == "" or texto.lower() in ("nan", "none"):
        return 0.0
    m = re.search(r"-?\d[\d.,]*", texto)
    if not m:
        return None
    n = m.group(0).rstrip(".,")
    if "," in n and "." in n:
        n = n.replace(",", "")
    elif "," in n:
        n = n.replace(",", ".")
    try:
        return float(n)
    except ValueError:
        return None


def calcular_beta(retornos_emisor, retornos_mercado):
    """
    Regresión lineal simple: retorno_emisor = alpha + beta * retorno_mercado.
    Devuelve un diccionario con beta, alpha, R², p-valor y n de obs. usadas,
    para que quede documentado qué tan confiable es cada beta.
    """
    df = pd.DataFrame({"emisor": retornos_emisor, "mercado": retornos_mercado}).dropna()
    n = len(df)
    if n < MIN_OBS_BETA:
        return {
            "beta": None, "alpha": None, "r2": None, "p_valor": None,
            "n_obs_beta": n,
            "nota": f"insuficientes observaciones superpuestas ({n} < {MIN_OBS_BETA})",
        }
    resultado = stats.linregress(df["mercado"], df["emisor"])
    return {
        "beta": round(resultado.slope, 6),
        "alpha": round(resultado.intercept, 6),
        "r2": round(resultado.rvalue ** 2, 4),
        "p_valor": round(resultado.pvalue, 6),
        "n_obs_beta": n,
        "nota": "",
    }


def registrar_log(lineas):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        for linea in lineas:
            f.write(linea + "\n")


# ----------------------------------------------------------------------------
# 2. EJECUCIÓN
# ----------------------------------------------------------------------------
def main():
    print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | scipy disponible")
    print(f"Leyendo crudo: {RUTA_CRUDO}\n")

    lineas_log = [
        "=" * 78,
        f"Corrida (limpieza): {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} "
        f"| Código de matrícula: {CODIGO_MATRICULA}",
    ]

    if not RUTA_CRUDO.exists():
        print(f"ERROR: no se encontró {RUTA_CRUDO}. Corre primero 01_extraccion_api.py.")
        lineas_log.append(f"Resultado: FALLO | no se encontró {RUTA_CRUDO}")
        registrar_log(lineas_log)
        return

    crudo = pd.read_csv(RUTA_CRUDO, parse_dates=["Date"])
    print(f"Crudo leído: {len(crudo)} filas | columnas: {list(crudo.columns)}\n")

    # ---- 2.1 Separar emisores de auxiliares (Categoria viene de 01) ----
    emisores = crudo[crudo["Categoria"] == "Emisor BVL"].copy()
    auxiliares = crudo[crudo["Categoria"] == "Auxiliar CAPM"].copy()

    faltan = set(["Categoria"]) - set(crudo.columns)
    if faltan:
        print("ERROR: el crudo no tiene columna 'Categoria'. ¿Es de una "
              "versión anterior de 01_extraccion_api.py? Vuelve a "
              "ejecutar 01 con la versión que incluye EPU y ^TNX.")
        lineas_log.append("Resultado: FALLO | crudo sin columna Categoria")
        registrar_log(lineas_log)
        return

    # ---- 2.2 Limpiar dividendos (solo en memoria, el crudo no se toca) ----
    antes_no_num = emisores["Dividends"].apply(
        lambda v: isinstance(v, str) and limpiar_dividendo(v) is None
    ).sum()
    emisores["Dividendo_efectivo"] = emisores["Dividends"].apply(limpiar_dividendo)
    if antes_no_num:
        print(f"AVISO: {antes_no_num} valores de Dividends no se pudieron "
              f"interpretar como número; revisa datos_procesados para esas filas.")
    lineas_log.append(f"Dividendos no interpretables: {antes_no_num}")

    # ---- 2.3 Retornos diarios por emisor y por EPU (llave: Date) ----
    emisores = emisores.sort_values(["Ticker", "Date"])
    emisores["Retorno_diario"] = emisores.groupby("Ticker")["Close"].pct_change()

    epu = auxiliares[auxiliares["Ticker"] == "EPU"].sort_values("Date").copy()
    epu["Retorno_EPU"] = epu["Close"].pct_change()
    epu_ret = epu[["Date", "Retorno_EPU"]]

    tnx = auxiliares[auxiliares["Ticker"] == "^TNX"].sort_values("Date").copy()
    # ^TNX ya es un rendimiento en %: Rf diario = Close / 100 (NO es un
    # retorno calculado con pct_change; ver docstring del módulo).
    tnx["Rf"] = tnx["Close"] / 100.0
    tnx_rf = tnx[["Date", "Rf"]]

    # ---- 2.4 Unir por la llave común (Date) ----
    panel = emisores.merge(epu_ret, on="Date", how="left")
    panel = panel.merge(tnx_rf, on="Date", how="left")

    n_sin_epu = panel["Retorno_EPU"].isna().sum()
    n_sin_rf = panel["Rf"].isna().sum()
    if n_sin_epu or n_sin_rf:
        print(f"AVISO: {n_sin_epu} filas sin Retorno_EPU y {n_sin_rf} filas "
              f"sin Rf tras la unión por fecha (días en que el emisor "
              f"cotizó pero EPU o ^TNX no, o viceversa).")
    lineas_log.append(f"Filas sin Retorno_EPU tras merge: {n_sin_epu}")
    lineas_log.append(f"Filas sin Rf tras merge: {n_sin_rf}")

    columnas_finales = [
        "Categoria", "Emisor", "Nemonico_BVL", "Ticker", "Date",
        "Open", "High", "Low", "Close", "Adj Close", "Volume",
        "Dividendo_efectivo", "Retorno_diario", "Retorno_EPU", "Rf",
    ]
    panel_final = panel[columnas_finales].copy()

    nombre_procesado = f"datos_procesados_{CODIGO_MATRICULA}.csv"
    ruta_procesado = DIR_PROCESADOS / nombre_procesado
    panel_final.to_csv(ruta_procesado, index=False, encoding="utf-8-sig")
    print(f"Guardado: {ruta_procesado} | {len(panel_final)} filas | "
          f"{len(panel_final.columns)} columnas")

    # ---- 2.5 Beta por emisor (regresión contra EPU) ----
    print("\nCalculando beta por emisor (regresión vs. EPU)...")
    filas_beta = []
    for ticker, grupo in panel_final.groupby("Ticker"):
        r = calcular_beta(grupo["Retorno_diario"], grupo["Retorno_EPU"])
        r["Ticker"] = ticker
        r["Emisor"] = grupo["Emisor"].iloc[0]
        r["Nemonico_BVL"] = grupo["Nemonico_BVL"].iloc[0]
        filas_beta.append(r)
        if r["beta"] is not None:
            estado = f"beta={r['beta']}"
        else:
            estado = f"SIN BETA ({r['nota']})"
        print(f"  {ticker:<14} {estado}")

    beta_df = pd.DataFrame(filas_beta)[
        ["Ticker", "Nemonico_BVL", "Emisor", "beta", "alpha", "r2",
         "p_valor", "n_obs_beta", "nota"]
    ]
    nombre_beta = f"beta_emisores_{CODIGO_MATRICULA}.csv"
    ruta_beta = DIR_PROCESADOS / nombre_beta
    beta_df.to_csv(ruta_beta, index=False, encoding="utf-8-sig")
    print(f"\nGuardado: {ruta_beta}")

    sin_beta = beta_df["beta"].isna().sum()
    if sin_beta:
        print(f"\nAVISO: {sin_beta} emisor(es) sin beta calculable. Revisa "
              f"{nombre_beta} antes de seguir a 04_analisis.py.")

    lineas_log.append(f"Resultado: EXITO | panel: {len(panel_final)} filas | "
                       f"beta calculado para {len(beta_df) - sin_beta}/"
                       f"{len(beta_df)} emisores")
    registrar_log(lineas_log)
    print(f"\nLog actualizado en: {LOG_PATH}")


if __name__ == "__main__":
    main()

Python 3.13.15 | pandas 2.2.3 | scipy disponible
Leyendo crudo: /content/datos_crudos/datos_crudos_2024200514G.csv

Crudo leído: 24059 filas | columnas: ['Categoria', 'Emisor', 'Nemonico_BVL', 'Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits', 'Capital Gains']

AVISO: 10 filas sin Retorno_EPU y 0 filas sin Rf tras la unión por fecha (días en que el emisor cotizó pero EPU o ^TNX no, o viceversa).
Guardado: /content/datos_procesados/datos_procesados_2024200514G.csv | 20037 filas | 15 columnas

Calculando beta por emisor (regresión vs. EPU)...
  ALICORC1.LM    beta=0.311458
  BACKUSI1.LM    beta=0.108912
  BAP            beta=0.902202
  BBVAC1.LM      beta=0.38556
  CPACASC1.LM    beta=0.224099
  CREDITC1.LM    beta=0.215367
  FERREYC1.LM    beta=0.395674
  LUSURC1.LM     beta=0.168888
  SCCO           beta=1.16989
  UNACEMC1.LM    beta=0.337374

Guardado: /content/datos_procesados/beta_emisores_2024200514G.csv

Log actualizado en: /conte

In [10]:
# Verificamos si R² y el p-valor de cada uno son confiables (beta emisores)
#Con eso se revisa emisor por emisor, si el beta es estadísticamente significativo
import pandas as pd
pd.set_option("display.width", 120)
beta = pd.read_csv("datos_procesados/beta_emisores_2024200514G.csv")
print(beta[["Ticker", "beta", "r2", "p_valor", "n_obs_beta"]].to_string(index=False))

     Ticker     beta     r2  p_valor  n_obs_beta
ALICORC1.LM 0.311458 0.0836 0.000000        2000
BACKUSI1.LM 0.108912 0.0109 0.000003        2002
        BAP 0.902202 0.4532 0.000000        2010
  BBVAC1.LM 0.385560 0.1235 0.000000        1999
CPACASC1.LM 0.224099 0.0299 0.000000        2001
CREDITC1.LM 0.215367 0.0295 0.000000        2001
FERREYC1.LM 0.395674 0.1135 0.000000        1999
 LUSURC1.LM 0.168888 0.0177 0.000000        2004
       SCCO 1.169890 0.5480 0.000000        2010
UNACEMC1.LM 0.337374 0.0692 0.000000        2001


In [16]:
#Para poder visualizar
import pandas as pd

ruta = "/content/datos_procesados/datos_procesados_2024200514G.csv"

df_procesado = pd.read_csv(ruta)

print("Filas y columnas:", df_procesado.shape)
print("Columnas:", df_procesado.columns.tolist())

df_procesado.head(3000)

Filas y columnas: (20037, 15)
Columnas: ['Categoria', 'Emisor', 'Nemonico_BVL', 'Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividendo_efectivo', 'Retorno_diario', 'Retorno_EPU', 'Rf']


,Categoria,Emisor,Nemonico_BVL,Ticker,Date,Open,High,Low,Close,Adj Close,Volume,Dividendo_efectivo,Retorno_diario,Retorno_EPU,Rf
0,Emisor BVL,Alicorp S.A.A.,ALICORC1,ALICORC1.LM,2018-01-02,10.610000,10.750000,10.610000,10.75,6.374446,27592,0.000000,NaN,NaN,0.02465
1,Emisor BVL,Alicorp S.A.A.,ALICORC1,ALICORC1.LM,2018-01-03,10.750000,10.750000,10.700000,10.70,6.344798,312206,0.000000,-0.004651,0.007891,0.02447
2,Emisor BVL,Alicorp S.A.A.,ALICORC1,ALICORC1.LM,2018-01-04,10.700000,10.850000,10.700000,10.85,6.433743,5757,0.000000,0.014019,0.002847,0.02453
3,Emisor BVL,Alicorp S.A.A.,ALICORC1,ALICORC1.LM,2018-01-05,10.890000,11.150000,10.890000,11.15,6.611635,516131,0.000000,0.027650,0.010173,0.02476
4,Emisor BVL,Alicorp S.A.A.,ALICORC1,ALICORC1.LM,2018-01-08,11.100000,11.200000,11.100000,11.20,6.641283,366278,0.000000,0.004484,-0.010773,0.02480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,Emisor BVL,Unión de Cervecerías Peruanas Backus y Johnston,BACKUSI1,BACKUSI1.LM,2021-12-15,22.260000,22.400000,22.100000,22.10,15.184223,10537,0.000000,-0.007188,-0.015597,0.01463
2996,Emisor BVL,Unión de Cervecerías Peruanas Backus y Johnston,BACKUSI1,BACKUSI1.LM,2021-12-16,22.290001,22.290001,21.400000,22.00,15.115520,28320,0.000000,-0.004525,0.023581,0.01422
2997,Emisor BVL,Unión de Cervecerías Peruanas Backus y Johnston,BACKUSI1,BACKUSI1.LM,2021-12-17,22.700001,22.900000,22.200001,22.90,15.733878,9472,0.000000,0.040909,0.012599,0.01402
2998,Emisor BVL,Unión de Cervecerías Peruanas Backus y Johnston,BACKUSI1,BACKUSI1.LM,2021-12-20,22.000000,22.000000,20.680000,20.68,15.703689,2048,2.180239,-0.096943,-0.014575,0.01419
